# Pneumonia detection v2 — Colab runner

Runs the full pipeline from this repo on a Colab GPU:
patient-level split → training → calibration → internal test + **external RSNA test** → Grad-CAM checks → results table.

**Before you start:** `Runtime → Change runtime type → T4 GPU` (or better).
Approximate time on a T4: data download + caching ~25 min, main model ~25 min, all four ablations ~1.5 h more.

> Colab can disconnect. Step 2 mounts Google Drive and writes all results there so nothing is lost.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Get the code

In [ ]:
!git clone -q https://github.com/empresst/PneumoniaDetectionExplainableAI.git
%cd PneumoniaDetectionExplainableAI
!pip install -q -e .

## 2. Save results to Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RESULTS = '/content/drive/MyDrive/pneumonia_v2'
!mkdir -p {RESULTS}/runs
!ln -sfn {RESULTS}/runs runs

## 3. Kaggle credentials

Upload your `kaggle.json` (Kaggle → Settings → API).
The cell below **never prints the file contents** — v1 of this project leaked the key exactly that way,
because `files.upload()` was the last line of a cell and Colab displayed its return value.

The RSNA download only works if this Kaggle account has accepted the
[competition rules](https://www.kaggle.com/competitions/rsna-pneumonia-detection-challenge/rules).

In [ ]:
import os
from google.colab import files

uploaded = files.upload()  # choose kaggle.json
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
with open(os.path.expanduser('~/.kaggle/kaggle.json'), 'wb') as f:
    f.write(uploaded['kaggle.json'])
os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)
del uploaded
print('kaggle.json saved (contents not shown)')

## 4. Download the datasets

In [ ]:
# Pediatric dataset (training + internal test), ~2.3 GB
!kaggle datasets download -q -d paultimothymooney/chest-xray-pneumonia -p data/raw
!unzip -q -o data/raw/chest-xray-pneumonia.zip -d data/raw/pediatric && rm data/raw/chest-xray-pneumonia.zip

# RSNA Pneumonia Detection Challenge (external test only), ~3.7 GB
!kaggle competitions download -q -c rsna-pneumonia-detection-challenge -p data/raw
!unzip -q -o data/raw/rsna-pneumonia-detection-challenge.zip 'stage_2_train_images/*' 'stage_2_train_labels.csv' 'stage_2_detailed_class_info.csv' -d data/raw/rsna
!rm data/raw/rsna-pneumonia-detection-challenge.zip

## 5. Build the manifest (patient-level split) and image cache

`--rsna-n 6000` takes a stratified sample of 6,000 RSNA studies to keep caching quick. Use `--rsna-n 0` for all ~26.7k.

In [ ]:
!python -m pneumo.prepare --pediatric data/raw/pediatric --rsna data/raw/rsna --rsna-n 6000 --out data --workers 4
!cp data/manifest.csv {RESULTS}/manifest.csv

## 6. Train, evaluate and explain the main model (DenseNet-121, 224 px)

In [ ]:
!python -m pneumo.train --config configs/default.yaml
!python -m pneumo.evaluate --run runs/densenet121_224
!python -m pneumo.explain --run runs/densenet121_224

In [ ]:
from IPython.display import Image, display
for subset in ['test', 'rsna_opacity_vs_normal']:
    display(Image(f'runs/densenet121_224/eval/{subset}/curves.png'))
    display(Image(f'runs/densenet121_224/eval/{subset}/gradcam_gallery.png'))

## 7. Baselines and ablations (optional, ~1.5 h)

| config | what it isolates |
|---|---|
| `ablation_mobilenetv2_frozen_128` | the v1 setup, re-run on the corrected split |
| `ablation_mobilenetv2_224` | effect of fine-tuning + resolution on the same backbone |
| `ablation_efficientnet_b0_224`, `ablation_resnet50_224` | backbone choice |

Runs already finished are saved on Drive, so after a disconnect you can re-run just the missing configs.

In [ ]:
!bash scripts/run_all.sh configs/ablation_mobilenetv2_frozen_128.yaml configs/ablation_mobilenetv2_224.yaml configs/ablation_efficientnet_b0_224.yaml configs/ablation_resnet50_224.yaml

## 8. Results table

In [ ]:
!python -m pneumo.summarize --runs runs --out {RESULTS}/results
from IPython.display import Markdown
display(Markdown(open(f'{RESULTS}/results/RESULTS.md').read()))

In [ ]:
import pandas as pd
pd.read_csv('runs/densenet121_224/eval/rsna_opacity_vs_all/subgroups.csv')

## 9. Bundle results (send this zip back for the README)

Model weights (`*.pt`) are excluded to keep it small; they stay on your Drive.

In [ ]:
!cd {RESULTS} && zip -q -r results_bundle.zip results runs -x '*.pt'
print(f'{RESULTS}/results_bundle.zip')

## 10. Interactive demo (optional)

In [ ]:
!pip install -q gradio
!python app.py --checkpoint runs/densenet121_224/best.pt --share